# MENA Biochar Feedstock-to-Market Optimisation Platform — V5 (evidence-integrated)

**Purpose.** Industrial screening platform linking feedstock supply → conversion route → product quality gates → market route → techno-economics → multi-objective optimisation.

**Important model-status statement:** this is a transparent decision-support prototype, not a bankable feasibility study or certification engine. The supplied research datasets mainly describe *biochar use in soils* and *incubation/permanence*, not a complete industrial manufacturing dataset. Country costs, prices, process yields, energy use, demand limits and stable-carbon fractions below are editable **scenario placeholders** until replaced with local quotations, plant data and verified LCA/MRV inputs. The notebook deliberately does not claim those placeholders are learned from the uploaded papers.

**V5 changes:** separate source audit; explicit MENA scenario configuration; technology-specific route eligibility; feedstock dry-mass and moisture balance; route-specific revenue/costs; lifecycle-emissions subtraction; Pareto frontier; uncertainty/sensitivity hooks; explicit lab/compliance hold points. HTC is labelled hydrochar, torrefaction is a pre-treatment/solid-fuel route, and gasification is not automatically treated as a biochar route.

In [1]:
from pathlib import Path
import os, re, json, math, warnings
import numpy as np
import pandas as pd
from itertools import product
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

# Finds files beside the notebook, in /mnt/data, or in common Kaggle paths.
SEARCH_ROOTS = [Path.cwd(), Path('/mnt/data'), Path('/kaggle/input')]
def find_source(filename):
    for root in SEARCH_ROOTS:
        if root.exists():
            hits = list(root.rglob(filename))
            if hits: return str(sorted(hits, key=lambda p:(len(str(p)),str(p)))[0])
    return None

SOURCE_FILES = {
    'biochar_ds': find_source('BiocharDS_V1.0.xlsx'),
    'incubation_2022_12': find_source('biochar_incubation_database_2022-12_SI4Manuscript.xlsx'),
    'incubation_live': find_source('biochar_incubation_database_2022-03-22_live.xlsx'),
    'incubation_backup': find_source('biochar_incubation_database_2022-03-22_backup_preUpdateHerathAubertin.xlsx'),
    'q10': find_source('biochar_q10_dataset_2022-05-27.xlsx'),
    'meta_analysis_xls': find_source('Biochar_Meta-analysis_ Database.xls'),
    'production_2024': find_source('Data_biochar_production_characteristics_202404.csv'),
    'bibliography_2024': find_source('Biblio_biochar_production_characteristics202404.csv'),
    'coproduct_heat': find_source('3-coproduct-use-model-heat-production.xlsx'),
    'puro_methodology': find_source('Puro Biochar Methodology - Edition 2025 (version 2) - For Publication.pdf'),
    'production_paper': find_source('A-critical-re-analysis-of-biochar-properties-prediction-from-production-parameters-and-elemental.pdf'),
}
pd.DataFrame([{'source':k,'path':v,'found':bool(v)} for k,v in SOURCE_FILES.items()])

,source,path,found
0,biochar_ds,/mnt/data/BiocharDS_V1.0.xlsx,True
1,incubation_2022_12,/mnt/data/biochar_incubation_database_2022-12_...,True
2,incubation_live,/mnt/data/biochar_incubation_database_2022-03-...,True
3,incubation_backup,/mnt/data/biochar_incubation_database_2022-03-...,True
4,q10,/mnt/data/biochar_q10_dataset_2022-05-27.xlsx,True
5,meta_analysis_xls,/mnt/data/Biochar_Meta-analysis_ Database.xls,True
6,production_2024,/mnt/data/Data_biochar_production_characterist...,True
7,bibliography_2024,/mnt/data/Biblio_biochar_production_characteri...,True
8,coproduct_heat,/mnt/data/3-coproduct-use-model-heat-productio...,True
9,puro_methodology,/mnt/data/Puro Biochar Methodology - Edition 2...,True


## 1. Source audit and evidence boundaries

The workbooks are not interchangeable tables. BiocharDS records outcomes of biochar application (e.g., soil and crop response); the incubation database records decomposition observations and metadata; the Q10 workbook contains temperature-response information. These are kept as separate evidence streams. The legacy `.xls` meta-analysis is loaded only if an Excel reader for that format is available.

In [2]:
def workbook_inventory(path, label):
    if not path: return []
    try:
        xls = pd.ExcelFile(path)
        out=[]
        for sh in xls.sheet_names:
            # Header=None preserves banner/header rows for these research workbooks.
            sample = pd.read_excel(path, sheet_name=sh, header=None, nrows=8)
            out.append({'source':label,'sheet':sh,'sample_rows':len(sample),
                        'sample_columns':sample.shape[1],
                        'first_row_preview':' | '.join(str(v) for v in sample.iloc[0,:min(8,sample.shape[1])].tolist())})
        return out
    except Exception as e:
        print(f'Could not inspect {label}: {type(e).__name__}: {e}')
        return [{'source':label,'sheet':'UNREADABLE','sample_rows':None,'sample_columns':None,'first_row_preview':str(e)}]

inventory=[]
for k,p in SOURCE_FILES.items(): inventory += workbook_inventory(p,k)
source_inventory = pd.DataFrame(inventory)
source_inventory

Could not inspect meta_analysis_xls: ImportError: Missing optional dependency 'xlrd'. Install xlrd >= 2.0.1 for xls Excel support Use pip or conda to install xlrd.
Could not inspect production_2024: ValueError: Excel file format cannot be determined, you must specify an engine manually.
Could not inspect bibliography_2024: ValueError: Excel file format cannot be determined, you must specify an engine manually.
Could not inspect puro_methodology: ValueError: Excel file format cannot be determined, you must specify an engine manually.
Could not inspect production_paper: ValueError: Excel file format cannot be determined, you must specify an engine manually.


,source,sheet,sample_rows,sample_columns,first_row_preview
0,biochar_ds,Readme,7.0,2.0,"Here, we constructed a database through 367 pe..."
1,biochar_ds,Sheet1,8.0,209.0,StudyID | Experiment ID | Author | Title | Yea...
2,biochar_ds,Explanation and unit,8.0,4.0,Information Overview | Target information and ...
3,biochar_ds,Reference,8.0,4.0,StudyID | Author | Title | Year_Publication
4,biochar_ds,Treatment,8.0,2.0,Treatment | Explanation
5,incubation_2022_12,articles,8.0,17.0,Table | Articles | 73 | articles | nan | nan |...
6,incubation_2022_12,data,8.0,27.0,Table | Data | nan | 136 | observations availa...
7,incubation_2022_12,metadata,8.0,66.0,Table | Metadata | nan | 181 | observations | ...
8,incubation_2022_12,validation,8.0,110.0,Table | Metadata_validation | nan | nan | nan ...
9,incubation_live,articles,8.0,17.0,Table | Articles | 73 | articles | nan | nan |...


### Load the directly usable BiocharDS table

The dataset's own Readme describes 367 peer-reviewed studies across 37 countries and 2,438 data items, with 21 indicators and background information. Its scope is biochar application effects—not industrial reactor yield or CAPEX. We use it for application-response evidence and metadata only.

In [3]:
biochar_ds = None
if SOURCE_FILES['biochar_ds']:
    biochar_ds = pd.read_excel(SOURCE_FILES['biochar_ds'], sheet_name='Sheet1')
    biochar_ds.columns = [str(x).strip() for x in biochar_ds.columns]
    print('BiocharDS shape:', biochar_ds.shape)
    display(biochar_ds.head(3))
else:
    print('BiocharDS not found. Place the workbook beside this notebook.')

# Basic schema/coverage audit; no silent assumptions about units or missingness.
if biochar_ds is not None:
    key_cols = [c for c in ['StudyID','Experiment ID','Country','Type_biochar','PyrolysisTemperature_biochar','C_biochar','Ash_biochar','pH_biochar','BiocharAddition','CropType','TrialDuration'] if c in biochar_ds.columns]
    ds_coverage = pd.DataFrame({'column':key_cols,
        'non_null_n':[biochar_ds[c].notna().sum() for c in key_cols],
        'non_null_pct':[100*biochar_ds[c].notna().mean() for c in key_cols]})
    display(ds_coverage)

BiocharDS shape: (2438, 209)


,StudyID,Experiment ID,Author,Title,Year_Publication,Country,Longitude,Latitude,ClimateZone,Temperature,Precipitation,WetnessIndex,SOC_initial,TN_initial,TP_initial,NH4_N_initial,NO3_N_initial,OlsenP_initial,pH_initial,BD_initial,CEC_initial,Texture_initial,Sand_initial,Silt _initial,Clay_initial,Type_biochar,ParticleSize_biochar,BD_biochar,PyrolysisTemperature_biochar,Ash_biochar,C_biochar,N_biochar,P_biochar,CNRatio_biochar,pH_biochar,CEC_biochar,Treatment,BiocharAddition,CropType,TrialDuration,SoilDepth,SOC_CK,SOC_CK_SD,SOC_CK_N,SOC_T,SOC_T _SD,SOC_T _N,SOC_RR,SOC_LNRR,TN_CK,...,SMBN_RR,SMBN_LNRR,CropYield_CK,CropYield_CK_SD,CropYield_CK_N,CropYield_T,CropYield_T _SD,CropYield_T _N,CropYield_RR,CropYield_LNRR,CO2_CK,CO2_CK_SD,CO2_CK_N,CO2_T,CO2_T _SD,CO2_T _N,CO2_RR,CO2_LNRR,CH4_CK,CH4_CK_SD,CH4_CK_N,CH4_T,CH4_T _SD,CH4_T _N,CH4_RR,CH4_LNRR,N2O_CK,N2O_CK_SD,N2O_CK_N,N2O_T,N2O_T _SD,N2O_T _N,N2O_RR,N2O_LNRR,GWP_CK,GWP_CK_SD,GWP_CK_N,GWP_T,GWP_T _SD,GWP_T _N,GWP_RR,GWP_LNRR,GHGI_CK,GHGI_CK_SD,GHGI_CK_N,GHGI_T,GHGI_T _SD,GHGI_T _N,GHGI_RR,GHGI_LNRR
0,1,1_1,A.O. Adekiya,Effects of biochar and poultry manure on soil ...,2019,Nigeria,6.016667,8.15,Tropic,32.0,NaN,NaN,9.16,1.6,NaN,NaN,NaN,NaN,5.61,1.49,NaN,Loamy sand,76.0,13.0,11.0,Wood,2.0,NaN,580,NaN,523.0,6.5,0.73,80.5,7.56,NaN,B,25.0,Radish,365.0,15.0,8.58,0.429,3.0,12.180000,0.609000,3.0,1.419580,0.350361,1.4,...,NaN,NaN,0.24,0.012,3.0,0.24,0.0120,3.0,1.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1_1,A.O. Adekiya,Effects of biochar and poultry manure on soil ...,2019,Nigeria,6.016667,8.15,Tropic,32.0,NaN,NaN,9.16,1.6,NaN,NaN,NaN,NaN,5.61,1.49,NaN,Loamy sand,76.0,13.0,11.0,Wood,2.0,NaN,580,NaN,523.0,6.5,0.73,80.5,7.56,NaN,B,50.0,Radish,365.0,15.0,8.58,0.429,3.0,18.560000,0.928000,3.0,2.163170,0.771575,1.4,...,NaN,NaN,0.24,0.012,3.0,0.32,0.0160,3.0,1.333333,0.287682,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,1_1,A.O. Adekiya,Effects of biochar and poultry manure on soil ...,2019,Nigeria,6.016667,8.15,Tropic,32.0,NaN,NaN,9.16,1.6,NaN,NaN,NaN,NaN,5.61,1.49,NaN,Loamy sand,76.0,13.0,11.0,Wood,2.0,NaN,580,NaN,523.0,6.5,0.73,80.5,7.56,NaN,BM,25.0,Radish,365.0,15.0,8.58,0.429,3.0,14.849188,0.742459,3.0,1.730675,0.548511,1.4,...,NaN,NaN,0.24,0.012,3.0,0.63,0.0315,3.0,2.625000,0.965081,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,column,non_null_n,non_null_pct
0,StudyID,2438,100.000000
1,Experiment ID,2438,100.000000
2,Country,2438,100.000000
3,Type_biochar,2301,94.380640
4,PyrolysisTemperature_biochar,2067,84.782609
5,C_biochar,1984,81.378179
6,Ash_biochar,526,21.575062
7,pH_biochar,2114,86.710418
8,BiocharAddition,2438,100.000000
9,CropType,2302,94.421657


### Incubation and Q10 data: keep the provenance visible

The incubation workbook has `articles`, `data`, `metadata`, and `validation` sheets. This cell loads those raw sheets without pretending their banner layout is a conventional rectangular table. The Q10 sheet is loaded separately. A validated key-based join and a decay-model refit are deliberately deferred until the observation/article identifiers and remaining-carbon fields have been verified against the workbook's validation notes.

In [4]:
INC_PATH = SOURCE_FILES['incubation_2022_12'] or SOURCE_FILES['incubation_live'] or SOURCE_FILES['incubation_backup']
inc_raw = {}
if INC_PATH:
    for sh in pd.ExcelFile(INC_PATH).sheet_names:
        inc_raw[sh] = pd.read_excel(INC_PATH, sheet_name=sh, header=None)
    print('Incubation release:', Path(INC_PATH).name)
    print({k:v.shape for k,v in inc_raw.items()})
if SOURCE_FILES['q10']:
    q10_raw = pd.read_excel(SOURCE_FILES['q10'], sheet_name='q10', header=None)
    print('Q10 raw shape:', q10_raw.shape)
else:
    q10_raw = None

# The legacy .xls parser is optional: do not silently install or substitute a different source.
meta_xls_status = 'not found'
if SOURCE_FILES['meta_analysis_xls']:
    try:
        meta_xls = pd.ExcelFile(SOURCE_FILES['meta_analysis_xls'])
        meta_xls_status = 'readable; sheets='+', '.join(meta_xls.sheet_names)
    except Exception as e:
        meta_xls_status = f'present but not readable in this environment ({type(e).__name__}: {e})'
print('Meta-analysis .xls status:', meta_xls_status)

Incubation release: biochar_incubation_database_2022-12_SI4Manuscript.xlsx
{'articles': (80, 17), 'data': (2945, 27), 'metadata': (191, 66), 'validation': (155, 110)}
Q10 raw shape: (34, 26)
Meta-analysis .xls status: present but not readable in this environment (ImportError: Missing optional dependency 'xlrd'. Install xlrd >= 2.0.1 for xls Excel support Use pip or conda to install xlrd.)


## 1A. 2024 production dataset and bibliography: schema, linkage and quality audit

The production CSV and bibliography CSV are a matched research release. `ID` is the publication-level key; it is not a unique experiment-row key. The dataset contains repeated observations within publications, so random row splitting would leak publication-specific information. All predictive validation below groups by `ID`. Missing target values are not imputed for training. The dataset contains 1,575 rows and 28 columns in the uploaded release; the paper describes a plausibility-screened analytical subset, so do not silently equate every CSV row with the paper's final modelling sample.

In [5]:
production = pd.read_csv(SOURCE_FILES['production_2024']) if SOURCE_FILES.get('production_2024') else None
bibliography = pd.read_csv(SOURCE_FILES['bibliography_2024']) if SOURCE_FILES.get('bibliography_2024') else None

if production is not None:
    production.columns = production.columns.astype(str).str.strip()
    print('Production data shape:', production.shape)
    print('Unique publication IDs:', production['ID'].nunique(), '| duplicate row count:', production.duplicated().sum())
    display(pd.DataFrame({'dtype':production.dtypes.astype(str), 'non_null_n':production.notna().sum(),
                          'missing_pct':(100*production.isna().mean()).round(1)}))
    display(production.groupby('Feedstock_class', dropna=False).size().sort_values(ascending=False).rename('rows').to_frame())
if bibliography is not None:
    bibliography.columns = bibliography.columns.astype(str).str.strip()
    print('Bibliography shape:', bibliography.shape, '| unique IDs:', bibliography['ID'].nunique())
    display(bibliography[['ID','Publication year','Title','DOI']].head())
if production is not None and bibliography is not None:
    ids_data=set(production['ID'].dropna().astype(str)); ids_bib=set(bibliography['ID'].dropna().astype(str))
    print('Data IDs without bibliography:', sorted(ids_data-ids_bib)[:20])
    print('Bibliography IDs without data:', sorted(ids_bib-ids_data)[:20])
    production_with_bib=production.merge(bibliography[['ID','Theme','Advanced characterization']], on='ID', how='left', validate='many_to_one')
    print('Rows successfully linked to bibliography:', production_with_bib['Theme'].notna().sum(), '/', len(production_with_bib))
else:
    production_with_bib=production

Production data shape: (1575, 28)
Unique publication IDs: 121 | duplicate row count: 0


,dtype,non_null_n,missing_pct
ID,int64,1575,0.0
Authors,object,1575,0.0
Title,object,1573,0.1
Publication year,float64,1567,0.5
Journal,object,1573,0.1
DOI,object,1542,2.1
Pyrolysis_type,object,1370,13.0
HTT,float64,1574,0.1
Pyrolysis_duration,object,1498,4.9
Heating_rate,float64,1174,25.5


,rows
Feedstock_class,
woody,427
herbaceous,375
shells and kernels,198
plastic,184
manure,114
kitchen waste,113
waste,106
sewage sludge,42
pure compound,15


Bibliography shape: (122, 14) | unique IDs: 122


,ID,Publication year,Title,DOI
0,1,2012.0,Effects of pyrolysis temperature on soybean st...,10.1016/j.biortech.2012.05.042
1,2,2014.0,Production and use of biochar from buffalo-wee...,10.1002/jctb.4157
2,3,1998.0,Obtention and surface characterisation of seve...,10.1016/S0008-6223(97)00237-6
3,4,2019.0,Comparative characterization of biochars produ...,10.7717/peerj.6784
4,5,2018.0,"Comparative analysis of physicochemical, nutri...",10.1007/s10163-017-0675-4


Data IDs without bibliography: []
Bibliography IDs without data: ['40']
Rows successfully linked to bibliography: 1560 / 1575


## 1B. Publication-grouped predictive baselines (research evidence, not universal process guarantees)

Train only on the uploaded measured rows, and only for targets with adequate sample count. The baseline uses temperature, residence time, heating rate, feedstock class and pyrolysis type. Validation is grouped by publication ID. Scores are reported as descriptive out-of-publication performance; they are not plant guarantees. A target with insufficient data or failed grouped CV is not exposed as a trusted predictor.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

MODEL_FEATURES_NUM = ['HTT','Pyrolysis_duration','Heating_rate']
MODEL_FEATURES_CAT = ['Feedstock_class','Pyrolysis_type']
MODEL_TARGETS = ['Yield','C','H','O','N','Ash','H_C','O_C']
production_model_scores=[]
production_models={}
if production is not None:
    feature_cols=[c for c in MODEL_FEATURES_NUM+MODEL_FEATURES_CAT if c in production.columns]
    for target in MODEL_TARGETS:
        if target not in production.columns: continue
        cols=feature_cols+[target,'ID']
        d=production[cols].copy()
        for col in MODEL_FEATURES_NUM:
            if col in d.columns: d[col]=pd.to_numeric(d[col],errors='coerce')
        d[target]=pd.to_numeric(d[target],errors='coerce')
        d=d.dropna(subset=[target,'ID'])
        n_groups=d['ID'].nunique()
        if len(d)<60 or n_groups<5:
            production_model_scores.append({'target':target,'n':len(d),'groups':n_groups,'status':'insufficient rows/groups'})
            continue
        numeric=[c for c in MODEL_FEATURES_NUM if c in d.columns]
        categorical=[c for c in MODEL_FEATURES_CAT if c in d.columns]
        prep=ColumnTransformer([
            ('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),numeric),
            ('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),categorical)
        ])
        model=Pipeline([('prep',prep),('rf',RandomForestRegressor(n_estimators=250,min_samples_leaf=3,max_features=0.8,random_state=42,n_jobs=-1))])
        X=d[numeric+categorical]; y=d[target]; groups=d['ID']
        cv=GroupKFold(n_splits=min(5,n_groups))
        try:
            pred=cross_val_predict(model,X,y,groups=groups,cv=cv,n_jobs=None)
            production_model_scores.append({'target':target,'n':len(d),'groups':n_groups,
                'MAE':mean_absolute_error(y,pred),'RMSE':mean_squared_error(y,pred)**0.5,
                'R2':r2_score(y,pred),'status':'grouped CV; inspect before deployment'})
            model.fit(X,y); production_models[target]={'model':model,'features':numeric+categorical}
        except Exception as e:
            production_model_scores.append({'target':target,'n':len(d),'groups':n_groups,'status':f'CV failed: {type(e).__name__}: {e}'})
    production_model_scores=pd.DataFrame(production_model_scores)
    display(production_model_scores)
else:
    print('2024 production CSV not found; predictive baselines skipped.')

# Explicit guardrail: these models are research baselines, not reactor guarantees.
def predict_research_property(target, input_row):
    if target not in production_models:
        raise ValueError(f'No validated research model available for target {target!r}')
    model_info=production_models[target]
    return float(model_info['model'].predict(pd.DataFrame([input_row])[model_info['features']])[0])

,target,n,groups,MAE,RMSE,R2,status
0,Yield,635,60,10.394878,15.964101,0.144836,grouped CV; inspect before deployment
1,C,1111,115,10.994108,14.592847,0.398349,grouped CV; inspect before deployment
2,H,1115,115,0.817839,1.151888,0.465772,grouped CV; inspect before deployment
3,O,1009,100,5.644381,7.493936,0.391384,grouped CV; inspect before deployment
4,N,1032,109,0.964297,2.056133,0.035728,grouped CV; inspect before deployment
5,Ash,1282,91,10.085986,13.865313,0.359578,grouped CV; inspect before deployment
6,H_C,1558,118,0.610774,4.358045,-0.004316,grouped CV; inspect before deployment
7,O_C,1530,117,0.151329,0.206358,-0.153363,grouped CV; inspect before deployment


## 1C. Coproduct heat/electricity model: source-derived equations

The uploaded workbook describes slow pyrolysis with direct combustion of pyrolysis gas/oil, drying, optional power generation and district-heat recovery. Its inputs include dry biomass LHV, as-received moisture, target moisture, dryer specific heat, drying-fan electricity, drying heat recovery fraction, biochar LHV/yield, electrical reactor heat input, boiler efficiency and power-to-heat ratio. This is not a generic model for fast pyrolysis, gasification, HTC or torrefaction. The calculator below mirrors the workbook's energy balance on a 1-tonne dry-biochar basis; it exposes assumptions rather than silently treating coproduct energy as free revenue.

In [7]:
coproduct_source_defaults = {
    'LHV_biomass_dry_MJ_kg':18.4, 'moisture_as_received':0.30, 'target_moisture':0.10,
    'dryer_specific_heat_MJ_kg_water':2.85, 'dryer_electricity_kWh_kg_water':0.05,
    'drying_heat_recovery_fraction':0.80, 'LHV_biochar_dry_MJ_kg':29.6,
    'biochar_yield_dry_basis':0.25, 'electric_heating_kWh_kg_biochar':0.0,
    'boiler_heat_efficiency':0.85, 'power_to_heat_ratio':0.15,
    'operation_electricity_kWh_kg_biochar':0.182
}
def slow_pyrolysis_coproduct_balance(inputs=None):
    x=coproduct_source_defaults.copy(); x.update(inputs or {})
    w=x['moisture_as_received']; wd=x['target_moisture']; y=x['biochar_yield_dry_basis']
    if not (0 <= wd < w < 1 and 0 < y <= 1):
        raise ValueError('Require 0 <= target moisture < as-received moisture < 1 and 0 < char yield <= 1')
    # Per 1 kg dry biochar, consistent with workbook rows 25-36.
    dry_biomass_kg=1/y
    water_removed_kg=(1/(1-w)-1/(1-wd))/y
    drying_duty_MJ=x['dryer_specific_heat_MJ_kg_water']*water_removed_kg
    drying_electricity_kWh=x['dryer_electricity_kWh_kg_water']*water_removed_kg
    pyrolysis_energy_MJ=(x['LHV_biomass_dry_MJ_kg']/y + x['electric_heating_kWh_kg_biochar']*3.6 - x['LHV_biochar_dry_MJ_kg'])
    useful_boiler_heat=max(0,pyrolysis_energy_MJ*x['boiler_heat_efficiency'])
    recovered_drying_heat=min(useful_boiler_heat,drying_duty_MJ)
    external_drying_heat=max(0,drying_duty_MJ-useful_boiler_heat)
    district_heat=max(0,useful_boiler_heat-drying_duty_MJ)+x['drying_heat_recovery_fraction']*drying_duty_MJ
    power_kWh=max(0, (pyrolysis_energy_MJ*x['boiler_heat_efficiency']*x['power_to_heat_ratio'])/3.6)
    return {'dry_biomass_kg_per_kg_char':dry_biomass_kg,'water_removed_kg_per_kg_char':water_removed_kg,
      'drying_duty_MJ_per_kg_char':drying_duty_MJ,'external_drying_heat_MJ_per_kg_char':external_drying_heat,
      'recovered_process_heat_to_dryer_MJ_per_kg_char':recovered_drying_heat,
      'district_heat_MJ_per_kg_char':district_heat,'electricity_generated_kWh_per_kg_char':power_kWh,
      'drying_electricity_kWh_per_kg_char':drying_electricity_kWh,
      'operation_electricity_kWh_per_kg_char':x['operation_electricity_kWh_kg_biochar'],
      'source':'3-coproduct-use-model-heat-production.xlsx; slow-pyrolysis-only; user must reconcile correlated inputs'}

display(pd.Series(slow_pyrolysis_coproduct_balance()))

dry_biomass_kg_per_kg_char                                                                      4.0
water_removed_kg_per_kg_char                                                               1.269841
drying_duty_MJ_per_kg_char                                                                 3.619048
external_drying_heat_MJ_per_kg_char                                                               0
recovered_process_heat_to_dryer_MJ_per_kg_char                                             3.619048
district_heat_MJ_per_kg_char                                                               36.67619
electricity_generated_kWh_per_kg_char                                                      1.558333
drying_electricity_kWh_per_kg_char                                                         0.063492
operation_electricity_kWh_per_kg_char                                                         0.182
source                                            3-coproduct-use-model-heat-production.xlsx; sl...


## 2. Editable MENA country scenarios

Values below are **illustrative placeholders** for building and testing the model interface. They are not current market quotations. Replace them with supplier quotes, energy tariffs, wage data, logistics bids, land/site costs, financing terms, demand/offtake evidence and jurisdiction-specific taxes before using outputs for investment decisions. The initial country is Egypt; other MENA scenarios are included as editable templates.

In [8]:
COUNTRY_SCENARIOS = {
    'Egypt':      dict(currency='EGP', fx_per_usd=50.0, electricity_usd_kwh=0.10, heat_fuel_usd_GJ=8.0, labour_usd_t_y=18.0, logistics_usd_t_km=0.08, feedstock_gate_usd_t=12.0, char_price_usd_t=180.0, carbon_price_usd_t=100.0, demand_char_t_y=15000),
    'Saudi Arabia':dict(currency='SAR', fx_per_usd=3.75, electricity_usd_kwh=0.08, heat_fuel_usd_GJ=6.0, labour_usd_t_y=30.0, logistics_usd_t_km=0.10, feedstock_gate_usd_t=25.0, char_price_usd_t=220.0, carbon_price_usd_t=100.0, demand_char_t_y=20000),
    'UAE':         dict(currency='AED', fx_per_usd=3.6725, electricity_usd_kwh=0.11, heat_fuel_usd_GJ=9.0, labour_usd_t_y=40.0, logistics_usd_t_km=0.12, feedstock_gate_usd_t=35.0, char_price_usd_t=260.0, carbon_price_usd_t=100.0, demand_char_t_y=12000),
    'Morocco':     dict(currency='MAD', fx_per_usd=10.0, electricity_usd_kwh=0.12, heat_fuel_usd_GJ=10.0, labour_usd_t_y=20.0, logistics_usd_t_km=0.07, feedstock_gate_usd_t=10.0, char_price_usd_t=170.0, carbon_price_usd_t=100.0, demand_char_t_y=10000),
    'Jordan':      dict(currency='JOD', fx_per_usd=0.709, electricity_usd_kwh=0.15, heat_fuel_usd_GJ=12.0, labour_usd_t_y=28.0, logistics_usd_t_km=0.09, feedstock_gate_usd_t=15.0, char_price_usd_t=190.0, carbon_price_usd_t=100.0, demand_char_t_y=5000),
    'Oman':        dict(currency='OMR', fx_per_usd=0.385, electricity_usd_kwh=0.10, heat_fuel_usd_GJ=8.0, labour_usd_t_y=32.0, logistics_usd_t_km=0.09, feedstock_gate_usd_t=18.0, char_price_usd_t=210.0, carbon_price_usd_t=100.0, demand_char_t_y=7000),
}

# Selected scenario and project definition — edit these values before running.
# Credit revenue remains disabled until a project-specific methodology/LCA/MRV assessment is independently reviewed.
CDR_CREDITING_ENABLED = False
CDR_ACCOUNTING_INPUTS_VERIFIED = False

COUNTRY = 'Egypt'
FEEDSTOCK = 'rice_husk'       # rice_husk, rice_straw, wheat_straw, cotton_stalk, date_palm, wood, manure
FEEDSTOCK_TPY_AS_RECEIVED = 10000.0
FEEDSTOCK_MOISTURE_WB = 0.12
FEEDSTOCK_TRANSPORT_KM = 60.0
PLANT_AVAILABILITY = 0.85
PLANT_LIFE_Y = 15
DISCOUNT_RATE = 0.12

SCENARIO = COUNTRY_SCENARIOS[COUNTRY].copy()
SCENARIO

{'currency': 'EGP',
 'fx_per_usd': 50.0,
 'electricity_usd_kwh': 0.1,
 'heat_fuel_usd_GJ': 8.0,
 'labour_usd_t_y': 18.0,
 'logistics_usd_t_km': 0.08,
 'feedstock_gate_usd_t': 12.0,
 'char_price_usd_t': 180.0,
 'carbon_price_usd_t': 100.0,
 'demand_char_t_y': 15000}

## 3. Feedstock and conversion assumptions

These are transparent engineering placeholders, not trained predictions. Replace with proximate/ultimate analysis, ash chemistry, particle size, bulk density, measured yields, gas composition, heat value, reactor vendor guarantees and pilot data. `hydrochar` is kept distinct from pyrolysis biochar; gasification and torrefaction do not automatically qualify for biochar/CDR routes.

In [9]:
FEEDSTOCKS = {
 'rice_husk':   dict(ash_frac=0.18, carbon_dry=0.40, nitrogen_dry=0.005, contamination=0.02),
 'rice_straw':  dict(ash_frac=0.17, carbon_dry=0.42, nitrogen_dry=0.006, contamination=0.02),
 'wheat_straw': dict(ash_frac=0.08, carbon_dry=0.45, nitrogen_dry=0.006, contamination=0.02),
 'cotton_stalk':dict(ash_frac=0.04, carbon_dry=0.46, nitrogen_dry=0.007, contamination=0.02),
 'date_palm':   dict(ash_frac=0.07, carbon_dry=0.45, nitrogen_dry=0.008, contamination=0.02),
 'wood':        dict(ash_frac=0.01, carbon_dry=0.50, nitrogen_dry=0.002, contamination=0.01),
 'manure':      dict(ash_frac=0.30, carbon_dry=0.36, nitrogen_dry=0.025, contamination=0.08),
}

# Illustrative dry-feed mass yields and energy demand. Must be replaced with measured/vendor data.
TECHNOLOGIES = {
 'slow_pyrolysis': dict(product='biochar', char_yield=0.32, process_heat_GJ_t_dry=1.6, electricity_kWh_t_dry=90, capex_usd_per_tpy=950, fixed_opex_fraction=0.06, char_C_fraction=0.68, stable_C_fraction=0.70, allowed_routes=['soil','construction','activation_precursor','cdr']),
 'fast_pyrolysis': dict(product='char_plus_bio_oil', char_yield=0.15, process_heat_GJ_t_dry=2.2, electricity_kWh_t_dry=130, capex_usd_per_tpy=1500, fixed_opex_fraction=0.07, char_C_fraction=0.72, stable_C_fraction=0.55, allowed_routes=['soil','construction','activation_precursor']),
 'flash_pyrolysis':dict(product='char_plus_vapours', char_yield=0.10, process_heat_GJ_t_dry=2.8, electricity_kWh_t_dry=170, capex_usd_per_tpy=1900, fixed_opex_fraction=0.08, char_C_fraction=0.75, stable_C_fraction=0.50, allowed_routes=['soil','construction']),
 'gasification':   dict(product='gas_and_ash_char', char_yield=0.08, process_heat_GJ_t_dry=1.3, electricity_kWh_t_dry=110, capex_usd_per_tpy=2100, fixed_opex_fraction=0.08, char_C_fraction=0.45, stable_C_fraction=0.30, allowed_routes=['energy_recovery']),
 'torrefaction':   dict(product='torrefied_solid', char_yield=0.70, process_heat_GJ_t_dry=0.8, electricity_kWh_t_dry=55, capex_usd_per_tpy=500, fixed_opex_fraction=0.05, char_C_fraction=0.58, stable_C_fraction=0.20, allowed_routes=['solid_fuel']),
 'hydrothermal_carbonisation':dict(product='hydrochar', char_yield=0.55, process_heat_GJ_t_dry=2.5, electricity_kWh_t_dry=100, capex_usd_per_tpy=1800, fixed_opex_fraction=0.08, char_C_fraction=0.55, stable_C_fraction=0.35, allowed_routes=['soil_screening','solid_fuel']),
}

ROUTES = {
 'soil': dict(price_key='char_price_usd_t', route_cost_usd_t=35, demand_fraction=1.0, needs_quality=True),
 'construction': dict(price_key='char_price_usd_t', route_cost_usd_t=55, demand_fraction=0.5, needs_quality=True),
 'activation_precursor': dict(price_key='char_price_usd_t', route_cost_usd_t=90, demand_fraction=0.15, needs_quality=True),
 'cdr': dict(price_key='char_price_usd_t', route_cost_usd_t=45, demand_fraction=0.5, needs_quality=True),
 'energy_recovery': dict(price_key=None, route_cost_usd_t=15, demand_fraction=1.0, needs_quality=False),
 'solid_fuel': dict(price_key=None, route_cost_usd_t=20, demand_fraction=1.0, needs_quality=False),
 'soil_screening': dict(price_key='char_price_usd_t', route_cost_usd_t=45, demand_fraction=0.2, needs_quality=True),
}

# Product-route prices are placeholder values; energy values are intentionally conservative.
ENERGY_VALUE_USD_GJ = 4.0
CARBON_CREDIT_PRICE_USD_TCO2 = SCENARIO['carbon_price_usd_t']

## 4. Industrial mass/energy/economic model

The prototype calculates dry-feed throughput, char mass, feedstock and transport cost, drying/process energy, electricity, annualised CAPEX, fixed/variable OPEX, route costs and indicative product revenue. It estimates *potential* durable carbon only for explicitly eligible routes and subtracts a simplified emissions proxy. This is **not** a full ISO-compliant LCA or carbon-credit issuance calculation: project-specific counterfactual, leakage, permanence methodology, supply-chain emissions, uncertainty deductions, monitoring and certification still need to be modelled.

In [10]:
def capital_recovery_factor(rate, years):
    return rate*(1+rate)**years/((1+rate)**years-1) if rate else 1/years

def evaluate_configuration(country, feedstock, feed_tpy_as_received, moisture_wb, transport_km,
                           technology, route, availability=0.85, char_price_override=None,
                           carbon_credit_price=None, capex_multiplier=1.0, energy_multiplier=1.0):
    if technology not in TECHNOLOGIES or feedstock not in FEEDSTOCKS or country not in COUNTRY_SCENARIOS:
        raise ValueError('Unknown country, feedstock or technology')
    tech=TECHNOLOGIES[technology]; fs=FEEDSTOCKS[feedstock]; market=COUNTRY_SCENARIOS[country]
    if route not in tech['allowed_routes']: return None
    if route not in ROUTES: return None
    if not (0 <= moisture_wb < 0.9): raise ValueError('Moisture must be a wet-basis fraction between 0 and 0.9')
    processed_feed_tpy_as_received = feed_tpy_as_received*availability
    dry_feed_tpy = processed_feed_tpy_as_received*(1-moisture_wb)
    water_per_dry_t = moisture_wb/max(1-moisture_wb,1e-9)
    # Approximate drying duty: 2.6 GJ per tonne water removed, with 70% thermal efficiency.
    drying_heat_GJ_t_dry = water_per_dry_t*2.6/0.70
    total_heat_GJ_t_dry = (drying_heat_GJ_t_dry+tech['process_heat_GJ_t_dry'])*energy_multiplier
    char_tpy = dry_feed_tpy*tech['char_yield']
    feed_cost = processed_feed_tpy_as_received*market['feedstock_gate_usd_t']
    transport_cost = processed_feed_tpy_as_received*transport_km*market['logistics_usd_t_km']
    electricity_cost = dry_feed_tpy*tech['electricity_kWh_t_dry']*market['electricity_usd_kwh']*energy_multiplier
    heat_cost = dry_feed_tpy*total_heat_GJ_t_dry*market['heat_fuel_usd_GJ']
    installed_capex = dry_feed_tpy*tech['capex_usd_per_tpy']*capex_multiplier
    annual_capital = installed_capex*capital_recovery_factor(DISCOUNT_RATE,PLANT_LIFE_Y)
    fixed_opex = installed_capex*tech['fixed_opex_fraction']
    labour = dry_feed_tpy*market['labour_usd_t_y']
    route_cfg=ROUTES[route]
    route_cost = char_tpy*route_cfg['route_cost_usd_t']
    if route_cfg['price_key']:
        unit_price = char_price_override if char_price_override is not None else market[route_cfg['price_key']]
        marketable_char_t = min(char_tpy, market['demand_char_t_y']*route_cfg['demand_fraction'])
        product_revenue = marketable_char_t*unit_price
    else:
        unit_price = ENERGY_VALUE_USD_GJ
        marketable_char_t = char_tpy
        product_revenue = char_tpy*unit_price
    # Potential durable carbon: not a credit claim; eligibility is route/technology screened only.
    eligible_for_cdr = route=='cdr' and tech['product']=='biochar'
    gross_stable_C_t = char_tpy*tech['char_C_fraction']*tech['stable_C_fraction'] if eligible_for_cdr else 0.0
    gross_stable_CO2_t = gross_stable_C_t*44/12
    # Screening emissions proxy: process heat + grid electricity + inbound logistics.
    # Replace factors with country-specific, source-specific LCA factors.
    heat_emissions = dry_feed_tpy*total_heat_GJ_t_dry*0.056
    grid_emissions = dry_feed_tpy*tech['electricity_kWh_t_dry']*0.00045
    logistics_emissions = processed_feed_tpy_as_received*transport_km*0.00008
    lifecycle_emissions_tCO2 = heat_emissions+grid_emissions+logistics_emissions
    net_removal_tCO2 = max(0.0,gross_stable_CO2_t-lifecycle_emissions_tCO2) if eligible_for_cdr else 0.0
    c_price = carbon_credit_price if carbon_credit_price is not None else market['carbon_price_usd_t']
    # Screening proxy is not creditable removal. Revenue is blocked unless both explicit gates are enabled.
    crediting_ready = bool(CDR_CREDITING_ENABLED and CDR_ACCOUNTING_INPUTS_VERIFIED)
    carbon_revenue = net_removal_tCO2*c_price if (eligible_for_cdr and crediting_ready) else 0.0
    total_cost = feed_cost+transport_cost+electricity_cost+heat_cost+annual_capital+fixed_opex+labour+route_cost
    revenue = product_revenue+carbon_revenue
    return {'country':country,'feedstock':feedstock,'technology':technology,'product_type':tech['product'],'route':route,
      'processed_feed_tpy_as_received':processed_feed_tpy_as_received,'dry_feed_tpy':dry_feed_tpy,'char_tpy':char_tpy,'marketable_char_tpy':marketable_char_t,
      'yield_dry_basis':tech['char_yield'],'gross_stable_CO2_tpy':gross_stable_CO2_t,
      'lifecycle_emissions_tCO2py':lifecycle_emissions_tCO2,'net_removal_tCO2py':net_removal_tCO2,
      'feedstock_cost_usd_y':feed_cost,'transport_cost_usd_y':transport_cost,'electricity_cost_usd_y':electricity_cost,
      'heat_cost_usd_y':heat_cost,'annualised_capex_usd_y':annual_capital,'fixed_opex_usd_y':fixed_opex,
      'labour_usd_y':labour,'route_cost_usd_y':route_cost,'product_revenue_usd_y':product_revenue,
      'carbon_revenue_usd_y':carbon_revenue,'total_cost_usd_y':total_cost,'total_revenue_usd_y':revenue,
      'annual_margin_usd_y':revenue-total_cost,'unit_product_price_usd_t':unit_price,
      'eligible_for_CDR_screen':eligible_for_cdr,'CDR_credit_revenue_enabled':crediting_ready,
      'net_removal_status':'SCREENING PROXY ONLY — NOT VERIFIED LCA/MRV','quality_status':'LAB TESTS REQUIRED — NOT CERTIFIED',
      'mass_balance_note':'Yield and energy values are editable scenario placeholders; coproduct balance is a separate slow-pyrolysis module.'}

# Quick smoke test: one scenario row, not an investment recommendation.
smoke = evaluate_configuration(COUNTRY, FEEDSTOCK, FEEDSTOCK_TPY_AS_RECEIVED, FEEDSTOCK_MOISTURE_WB,
                               FEEDSTOCK_TRANSPORT_KM, 'slow_pyrolysis', 'cdr', PLANT_AVAILABILITY)
pd.Series(smoke)

country                                                                       Egypt
feedstock                                                                 rice_husk
technology                                                           slow_pyrolysis
product_type                                                                biochar
route                                                                           cdr
processed_feed_tpy_as_received                                               8500.0
dry_feed_tpy                                                                 7480.0
char_tpy                                                                     2393.6
marketable_char_tpy                                                          2393.6
yield_dry_basis                                                                0.32
gross_stable_CO2_tpy                                                    4177.629867
lifecycle_emissions_tCO2py                                                 1

## 5. Product application gates (screening, not certification)

A model-predicted carbon fraction, H/C ratio or pH cannot establish product compliance. Each route needs independent analytical and performance evidence. The screen below states the missing tests and prevents the notebook from labelling any product “certified”. Activation is a separate process step; activated-carbon economics and adsorption properties are not inferred from biochar alone.

In [11]:
APPLICATION_REQUIREMENTS = {
 'soil': ['H/Corg and O/Corg; PAHs; heavy metals; PCBs/dioxins where relevant; pH; EC; ash; nutrients; pathogen/contaminant screen; agronomic field validation'],
 'construction': ['particle size; moisture; ash/mineralogy; compressive strength; durability; leaching; fire/smoke behaviour; compatibility with binder; standards testing'],
 'activation_precursor': ['activation route and yield; BET surface area; pore-size distribution; iodine/methylene-blue adsorption; abrasion/hardness; chemical/water use; effluent treatment'],
 'cdr': ['approved methodology; feedstock sustainability; chain of custody; net lifecycle removal; stability/permanence; leakage; MRV; independent validation/verification'],
 'energy_recovery': ['LHV; ash fusion; emissions; particulate/NOx/SOx; operating permits; boiler/gasifier compatibility'],
 'solid_fuel': ['LHV; moisture; ash; durability; emissions; applicable fuel/product standards'],
 'soil_screening': ['hydrochar-specific contaminant and agronomic testing; soil/ecotoxicity assessment; jurisdictional approval'],
}
def application_gate(technology, route):
    tech=TECHNOLOGIES[technology]
    eligible = route in tech['allowed_routes']
    if route=='cdr' and tech['product']!='biochar': eligible=False
    return {'route_eligible_by_model':eligible,
            'certification_claim_allowed':False,
            'status':'ELIGIBLE FOR FURTHER TESTING' if eligible else 'NOT ELIGIBLE IN CURRENT SCREEN',
            'required_tests':APPLICATION_REQUIREMENTS.get(route,[])}
application_gate('slow_pyrolysis','cdr')

{'route_eligible_by_model': True,
 'certification_claim_allowed': False,
 'status': 'ELIGIBLE FOR FURTHER TESTING',
 'required_tests': ['approved methodology; feedstock sustainability; chain of custody; net lifecycle removal; stability/permanence; leakage; MRV; independent validation/verification']}

## 6. Multi-objective optimisation and Pareto frontier

The optimizer compares economic margin and net CO₂ removal. It does not collapse them into an arbitrary weighted score. Routes are only generated where the technology table permits them. Each row assigns the full char output to one route; split allocation is not yet represented. The Pareto set contains configurations for which no other configuration is at least as good in both objectives and strictly better in one. The current grid is intentionally modest and should be expanded after the scenario assumptions are calibrated.

In [12]:
def run_optimisation(country=COUNTRY, feedstock=FEEDSTOCK, feed_tpy=FEEDSTOCK_TPY_AS_RECEIVED,
                     moisture=FEEDSTOCK_MOISTURE_WB, transport_km=FEEDSTOCK_TRANSPORT_KM,
                     availability=PLANT_AVAILABILITY, capex_multiplier=1.0, energy_multiplier=1.0):
    rows=[]
    for tech_name, tech in TECHNOLOGIES.items():
        for route in tech['allowed_routes']:
            # No double-counting: each row allocates the entire char output to one route.
            row=evaluate_configuration(country,feedstock,feed_tpy,moisture,transport_km,
                 tech_name,route,availability,capex_multiplier=capex_multiplier,energy_multiplier=energy_multiplier)
            if row: rows.append(row)
    df=pd.DataFrame(rows)
    if df.empty: return df,df
    # Pareto: maximise margin and net removal.
    vals=df[['annual_margin_usd_y','net_removal_tCO2py']].to_numpy()
    is_pareto=np.ones(len(df),dtype=bool)
    for i,(m,c) in enumerate(vals):
        dominated=(vals[:,0]>=m)&(vals[:,1]>=c)&((vals[:,0]>m)|(vals[:,1]>c))
        if dominated.any(): is_pareto[i]=False
    df['pareto_optimal']=is_pareto
    return df.sort_values(['pareto_optimal','annual_margin_usd_y'],ascending=[False,False]), df[df.pareto_optimal].sort_values('annual_margin_usd_y',ascending=False)

results, pareto = run_optimisation()
print('Scenarios evaluated:',len(results),'| Pareto solutions:',len(pareto))
display(results[['technology','product_type','route','dry_feed_tpy','char_tpy','annual_margin_usd_y','net_removal_tCO2py','pareto_optimal','quality_status']])

Scenarios evaluated: 13 | Pareto solutions: 2


,technology,product_type,route,dry_feed_tpy,char_tpy,annual_margin_usd_y,net_removal_tCO2py,pareto_optimal,quality_status
10,torrefaction,torrefied_solid,solid_fuel,7480.0,5236.0,-1.216659e+06,0.000000,True,LAB TESTS REQUIRED — NOT CERTIFIED
3,slow_pyrolysis,biochar,cdr,7480.0,2393.6,-1.617370e+06,2951.521867,True,LAB TESTS REQUIRED — NOT CERTIFIED
0,slow_pyrolysis,biochar,soil,7480.0,2393.6,-1.593434e+06,0.000000,False,LAB TESTS REQUIRED — NOT CERTIFIED
1,slow_pyrolysis,biochar,construction,7480.0,2393.6,-1.641306e+06,0.000000,False,LAB TESTS REQUIRED — NOT CERTIFIED
2,slow_pyrolysis,biochar,activation_precursor,7480.0,2393.6,-1.750930e+06,0.000000,False,LAB TESTS REQUIRED — NOT CERTIFIED
4,fast_pyrolysis,char_plus_bio_oil,soil,7480.0,1122.0,-2.806715e+06,0.000000,False,LAB TESTS REQUIRED — NOT CERTIFIED
5,fast_pyrolysis,char_plus_bio_oil,construction,7480.0,1122.0,-2.829155e+06,0.000000,False,LAB TESTS REQUIRED — NOT CERTIFIED
6,fast_pyrolysis,char_plus_bio_oil,activation_precursor,7480.0,1122.0,-2.868425e+06,0.000000,False,LAB TESTS REQUIRED — NOT CERTIFIED
11,hydrothermal_carbonisation,hydrochar,soil_screening,7480.0,4114.0,-3.231240e+06,0.000000,False,LAB TESTS REQUIRED — NOT CERTIFIED
12,hydrothermal_carbonisation,hydrochar,solid_fuel,7480.0,4114.0,-3.651934e+06,0.000000,False,LAB TESTS REQUIRED — NOT CERTIFIED


## 7. Sensitivity and uncertainty hooks

The next cell varies feedstock gate price, carbon price, CAPEX and energy cost multipliers. It is a deterministic scenario envelope—not a statistical confidence interval. Once measured distributions and vendor quotations are available, replace the ranges with Monte Carlo sampling and propagate uncertainty through yield, stable carbon, LCA and price/demand.

In [13]:
def sensitivity_analysis():
    rows=[]
    for feed_cost_mult, carbon_mult, capex_mult, energy_mult in product([0.7,1.0,1.3],[0.0,1.0,1.5],[0.8,1.0,1.3],[0.8,1.0,1.25]):
        scenario=COUNTRY_SCENARIOS[COUNTRY].copy()
        scenario['feedstock_gate_usd_t']*=feed_cost_mult
        scenario['carbon_price_usd_t']*=carbon_mult
        # Evaluate eligible CDR and soil/market cases against a temporary scenario copy.
        old=COUNTRY_SCENARIOS[COUNTRY]
        COUNTRY_SCENARIOS[COUNTRY]=scenario
        try:
            df,_=run_optimisation(capex_multiplier=capex_mult,energy_multiplier=energy_mult)
        finally:
            COUNTRY_SCENARIOS[COUNTRY]=old
        if len(df):
            best=df.iloc[0]
            rows.append({'feedstock_cost_multiplier':feed_cost_mult,'carbon_price_multiplier':carbon_mult,
                         'capex_multiplier':capex_mult,'energy_multiplier':energy_mult,
                         'best_technology':best.technology,'best_route':best.route,
                         'best_margin_usd_y':best.annual_margin_usd_y,'net_removal_tCO2py':best.net_removal_tCO2py})
    return pd.DataFrame(rows)

sensitivity = sensitivity_analysis()
display(sensitivity.head(12))

,feedstock_cost_multiplier,carbon_price_multiplier,capex_multiplier,energy_multiplier,best_technology,best_route,best_margin_usd_y,net_removal_tCO2py
0,0.7,0.0,0.8,0.80,torrefaction,solid_fuel,-1.014971e+06,0.0
1,0.7,0.0,0.8,1.00,torrefaction,solid_fuel,-1.038835e+06,0.0
2,0.7,0.0,0.8,1.25,torrefaction,solid_fuel,-1.068665e+06,0.0
3,0.7,0.0,1.0,0.80,torrefaction,solid_fuel,-1.162195e+06,0.0
4,0.7,0.0,1.0,1.00,torrefaction,solid_fuel,-1.186059e+06,0.0
5,0.7,0.0,1.0,1.25,torrefaction,solid_fuel,-1.215889e+06,0.0
6,0.7,0.0,1.3,0.80,torrefaction,solid_fuel,-1.383032e+06,0.0
7,0.7,0.0,1.3,1.00,torrefaction,solid_fuel,-1.406896e+06,0.0
8,0.7,0.0,1.3,1.25,torrefaction,solid_fuel,-1.436726e+06,0.0
9,0.7,1.0,0.8,0.80,torrefaction,solid_fuel,-1.014971e+06,0.0


## 8. Interpretation, limitations and required next data

### What this V4 prototype can do
- Audit the uploaded research workbooks and preserve their separate scopes.
- Compare editable country/feedstock/technology/route scenarios.
- Produce transparent annual mass, cost, revenue and indicative carbon-removal calculations.
- Identify Pareto-optimal configurations and expose sensitivity to major assumptions.
- List route-specific tests and hold points instead of claiming regulatory certification.

### What it cannot yet support
- Bankable CAPEX/OPEX, guaranteed yield, plant sizing or lender-grade IRR/NPV.
- Certified biochar, activated carbon, construction additive or carbon-credit claims.
- Country-specific market demand or prices without validated buyer/offtake evidence.
- Reliable ML predictions of manufacturing yield/property outcomes: the supplied datasets do not constitute a complete, harmonised process-to-product training set.
- A full lifecycle assessment or carbon-credit methodology calculation. Carbon-credit revenue is disabled by default; the reported net-removal figure is a simplified screening proxy only. The uploaded Puro 2025 methodology requires project-specific eligibility, baseline/additionality, supply-chain and use-phase accounting, leakage/indirect-effect assessment, monitoring, measurement, uncertainty treatment and verification; this notebook does not implement or certify those requirements.

### Data needed to advance beyond V5
1. Feedstock-by-country annual availability, seasonality, moisture, ash chemistry, contaminants, collection radius and gate price.
2. Vendor or pilot mass/energy balances by reactor type and scale, including gas/oil/char yields and heat recovery.
3. Country-specific electricity/fuel tariffs, labour, land, water, permitting, taxes, financing and logistics quotations.
4. Buyer specifications, verified demand/offtake, delivered prices and product qualification requirements for each application.
5. Lab data for proximate/ultimate analysis, H/Corg, O/Corg, PAHs, metals, SSA/adsorption, leaching and application-specific performance.
6. A selected carbon-removal methodology with counterfactual, LCA factors, permanence, leakage, MRV and verification costs.

**Decision rule:** treat every numeric output as a scenario result until the placeholder inputs are replaced and independently checked. Do not use the present notebook as an investment recommendation.

## 9. Carbon methodology and evidence boundary (Puro 2025 upload)

The uploaded *Puro Biochar Methodology — Edition 2025, version 2* is treated as a methodology reference, not as evidence that a project is eligible or credits can be issued. A project assessment must establish feedstock eligibility and sustainability, chain of custody, the counterfactual/baseline and additionality, project boundary, process and transport emissions, end-use/storage conditions, permanence/stability, leakage and indirect effects, monitoring plan, sampling and analytical QA/QC, uncertainty deductions, reporting and independent verification. The exact applicable methodology version and any supplementary rules must be checked at project registration. This notebook has not encoded all methodology clauses or performed a validation/verification audit.

**Carbon accounting rule in this prototype:** `net_removal_tCO2py` is a simplified scenario screening proxy based on assumed char carbon/stability fractions less coarse heat/grid/logistics emission factors. It is not a Puro-issued removal quantity, not a creditable quantity, and must not be marketed as such. Carbon-credit revenue is disabled unless both `CDR_CREDITING_ENABLED` and `CDR_ACCOUNTING_INPUTS_VERIFIED` are deliberately set to `True` after independent project review. That switch is a user-controlled guardrail, not an automated compliance determination.

## 10. Source manifest and dataset-specific cautions

| Uploaded source | Role in this notebook | Important limitation |
|---|---|---|
| `Data_biochar_production_characteristics_202404.csv` | Process-condition/property research data; publication-grouped baselines | Large missingness; not a plant/vendor dataset; publication-level dependence |
| `Biblio_biochar_production_characteristics202404.csv` | Publication metadata joined by `ID` | 122 bibliography rows; verify IDs and source records |
| `A-critical-re-analysis-of-biochar-properties-prediction-from-production-parameters-and-elemental.pdf` | Scientific interpretation and data-quality caveats | Findings depend on paper's screened subset and modelling protocol |
| `3-coproduct-use-model-heat-production.xlsx` | Slow-pyrolysis gas/oil combustion, drying, power and heat balance | Not generic across conversion technologies; inputs are correlated |
| `Puro Biochar Methodology - Edition 2025 (version 2) - For Publication.pdf` | CDR eligibility, accounting, MRV and verification reference | Methodology compliance must be established project-by-project |
| `BiocharDS_V1.0.xlsx` | Application/soil response evidence | Not reactor-yield or manufacturing economics data |
| Incubation database releases and Q10 workbook | Persistence/decomposition evidence | Distinct experimental designs and units; no automatic pooling |

**Model governance:** every input should be tagged as (1) measured project data, (2) literature-derived, (3) vendor-guaranteed, or (4) placeholder. Keep this provenance tag alongside the value in the production deployment. A high-performing grouped-CV model is still only a research predictor until independently validated on representative pilot/plant data.